# Model Evaluation & Decision Threshold Tuning

### What Problem Does This Notebook Solve?
When a classification model evaluates a sample, it outputs a probability (e.g., "70% chance of churn"). 
By default, Scikit-Learn applies a hardcoded cutoff of **0.50**:
* Probability $\ge 0.50 \rightarrow$ Positive class (1)
* Probability $< 0.50 \rightarrow$ Negative class (0)

In the real world, **0.50 is rarely the optimal cutoff**:
* In fraud detection or medical screening, waiting for 50% confidence causes missed cases (high cost of False Negatives). You might prefer flagging at 0.20.
* In customer retention campaigns, you want the exact probability cutoff that maximizes revenue and minimizes wasted discounts.

### What We Will Do (Purely Numerical):
1. **Fit a model** and extract both default predictions (`predict()`) and continuous probabilities (`predict_proba()`).
2. **Diagnose the default 0.50 cutoff** using raw confusion matrices and classification metrics.
3. **Inspect the Precision vs. Recall tradeoff numerically** across cutoffs from 0.10 to 0.80.
4. **Optimize the threshold against business profit** using a cost-benefit calculation.
5. **Apply the custom threshold** in production code.
6. **Check probability reliability** using the Brier score.

In [11]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)

np.random.seed(42)

# 1. Synthesize an imbalanced customer dataset (88% retain, 12% churn)
X_raw, y_raw = make_classification(
    n_samples=2500,
    n_features=10,
    n_informative=6,
    weights=[0.88, 0.12],
    random_state=42
)

X = pd.DataFrame(X_raw, columns=[f"feature_{i}" for i in range(10)])
y = pd.Series(y_raw, name="churn")

# 2. Split into Train (80%) and Test (20%)
# The test set remains untouched until evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Fit classifier
model = HistGradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)

# 4. Extract default binary predictions (0.50 cutoff) AND raw probabilities
y_pred_default = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"Test Set Total: {len(y_test)} rows")
print(f"Actual Churners: {y_test.sum()} | Actual Retained: {len(y_test) - y_test.sum()}")

Test Set Total: 500 rows
Actual Churners: 61 | Actual Retained: 439


---
## Step 1: Evaluating the Default 0.50 Cutoff

We evaluate the baseline predictions produced by `.predict()`.

### Metrics to Track:
* **Confusion Matrix:** Shows raw counts of True Negatives, False Positives, False Negatives, and True Positives.
* **ROC-AUC:** Overall ranking ability across all thresholds (evaluates both classes).
* **PR-AUC (`average_precision`):** Overall ranking ability focused strictly on the minority (churn) class.

In [15]:
print("=== CONFUSION MATRIX (DEFAULT 0.50 THRESHOLD) ===")
cm_default = confusion_matrix(y_test, y_pred_default)
df_cm_default = pd.DataFrame(
    cm_default, 
    index=['Actual Retained (0)', 'Actual Churned (1)'], 
    columns=['Pred Retained (0)', 'Pred Churned (1)']
)
display(df_cm_default)

print("\n=== CLASSIFICATION REPORT (DEFAULT 0.50 THRESHOLD) ===")
print(classification_report(y_test, y_pred_default, target_names=['Retained (0)', 'Churned (1)']))

print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")
print(f"PR-AUC Score:  {average_precision_score(y_test, y_proba):.4f}")

=== CONFUSION MATRIX (DEFAULT 0.50 THRESHOLD) ===


,Pred Retained (0),Pred Churned (1)
Actual Retained (0),437,2
Actual Churned (1),15,46



=== CLASSIFICATION REPORT (DEFAULT 0.50 THRESHOLD) ===
              precision    recall  f1-score   support

Retained (0)       0.97      1.00      0.98       439
 Churned (1)       0.96      0.75      0.84        61

    accuracy                           0.97       500
   macro avg       0.96      0.87      0.91       500
weighted avg       0.97      0.97      0.96       500

ROC-AUC Score: 0.9750
PR-AUC Score:  0.9286


---
## Step 2: The Precision vs. Recall Tradeoff (Numeric Inspection)

Instead of relying on a graph, we sweep probability cutoffs from 0.10 to 0.80 and display the exact results in a table.

* **Lower threshold ($\le 0.20$):** We flag anyone with even a slight chance of churn. Recall rises (we catch almost all churners), but Precision drops (more false alarms).
* **Higher threshold ($\ge 0.70$):** We only flag customers we are positive will churn. Precision rises (few false alarms), but Recall drops (we miss many actual churners).

In [16]:
cutoffs = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
tradeoff_results = []

for c in cutoffs:
    preds = (y_proba >= c).astype(int)
    tradeoff_results.append({
        'Threshold': c,
        'Precision': round(precision_score(y_test, preds, zero_division=0), 3),
        'Recall': round(recall_score(y_test, preds), 3),
        'F1-Score': round(f1_score(y_test, preds, zero_division=0), 3),
        'Total Flagged': preds.sum()
    })

df_tradeoff = pd.DataFrame(tradeoff_results)
print("=== PRECISION vs. RECALL TRADEOFF TABLE ===")
display(df_tradeoff)

=== PRECISION vs. RECALL TRADEOFF TABLE ===


,Threshold,Precision,Recall,F1-Score,Total Flagged
0,0.1,0.824,0.918,0.868,68
1,0.2,0.867,0.852,0.860,60
2,0.3,0.891,0.803,0.845,55
3,0.4,0.904,0.770,0.832,52
4,0.5,0.958,0.754,0.844,48
5,0.6,0.978,0.738,0.841,46
6,0.7,1.000,0.721,0.838,44
7,0.8,1.000,0.672,0.804,41


---
## Step 3: Finding the Business-Optimal Threshold

In production, thresholds are driven by economics rather than arbitrary defaults.

### Business Scenario (Customer Retention Program):
* **True Positive (TP):** We catch a true churner and save them. Net gain = **+$150**.
* **False Positive (FP):** We send an unnecessary discount to a happy customer. Net loss = **-$25**.
* **False Negative (FN):** We miss an actual churner and they leave. Net loss = **-$300**.
* **True Negative (TN):** We leave a happy customer alone. Net value = **$0**.

We test 100 cutoffs between 0.05 and 0.95 to find the exact threshold that yields maximum net profit.

In [17]:
def calculate_profit(y_true, probs, cutoff, val_tp=150, cost_fp=-25, cost_fn=-300, val_tn=0):
    preds = (probs >= cutoff).astype(int)
    tp = np.sum((y_true == 1) & (preds == 1))
    fp = np.sum((y_true == 0) & (preds == 1))
    fn = np.sum((y_true == 1) & (preds == 0))
    tn = np.sum((y_true == 0) & (preds == 0))
    return (tp * val_tp) + (fp * cost_fp) + (fn * cost_fn) + (tn * val_tn)

# Test 100 possible cutoff levels
tested_cutoffs = np.linspace(0.05, 0.95, 100)
profit_list = [calculate_profit(y_test, y_proba, t) for t in tested_cutoffs]

best_idx = np.argmax(profit_list)
optimal_threshold = tested_cutoffs[best_idx]
max_profit = profit_list[best_idx]

# Compare against the default 0.50 cutoff
profit_default = calculate_profit(y_test, y_proba, 0.50)

print("=== BUSINESS FINANCIAL OPTIMIZATION ===")
print(f"Default Cutoff (0.50) Profit: ${profit_default:,.2f}")
print(f"Optimal Cutoff ({optimal_threshold:.2f}) Profit: ${max_profit:,.2f}")
print(f"Net Financial Improvement:    +${max_profit - profit_default:,.2f}")

=== BUSINESS FINANCIAL OPTIMIZATION ===
Default Cutoff (0.50) Profit: $2,350.00
Optimal Cutoff (0.09) Profit: $6,600.00
Net Financial Improvement:    +$4,250.00


---
## Step 4: Applying the Tuned Threshold to Make Decisions

Models do not store a threshold parameter internally. In production, we extract probabilities using `predict_proba()` and apply our chosen cutoff manually.

In [18]:
# Apply optimal cutoff
y_pred_tuned = (y_proba >= optimal_threshold).astype(int)

cm_tuned = confusion_matrix(y_test, y_pred_tuned)
df_cm_tuned = pd.DataFrame(
    cm_tuned, 
    index=['Actual Retained (0)', 'Actual Churned (1)'], 
    columns=['Pred Retained (0)', 'Pred Churned (1)']
)

print(f"=== CONFUSION MATRIX AT OPTIMAL THRESHOLD ({optimal_threshold:.2f}) ===")
display(df_cm_tuned)

# Compare detection rates
churners_caught_default = cm_default[1, 1]
churners_caught_tuned = cm_tuned[1, 1]
total_churners = y_test.sum()

print("\n=== SUMMARY COMPARISON ===")
print(f"Total Churners in Test Data: {total_churners}")
print(f"Churners Captured at 0.50:  {churners_caught_default} ({(churners_caught_default/total_churners):.1%})")
print(f"Churners Captured at {optimal_threshold:.2f}:  {churners_caught_tuned} ({(churners_caught_tuned/total_churners):.1%})")

=== CONFUSION MATRIX AT OPTIMAL THRESHOLD (0.09) ===


,Pred Retained (0),Pred Churned (1)
Actual Retained (0),427,12
Actual Churned (1),5,56



=== SUMMARY COMPARISON ===
Total Churners in Test Data: 61
Churners Captured at 0.50:  46 (75.4%)
Churners Captured at 0.09:  56 (91.8%)


---
## Step 5: Checking Probability Calibration (Brier Score)

Threshold tuning relies on model confidence being truthful.
* If a model outputs a probability of `0.70`, items with that score should convert roughly 70% of the time.
* Some models (like uncalibrated tree ensembles) can produce skewed confidence scores.

### Numerical Check: Brier Score Loss
Measures the mean squared difference between predicted probabilities and actual 0/1 outcomes:
* **0.00:** Perfect calibration (probabilities match reality exactly).
* **Low (< 0.10):** Well-calibrated, reliable probabilities.
* **High (> 0.20):** Distorted probabilities that should not be interpreted as true percentages.

In [19]:
brier_score = brier_score_loss(y_test, y_proba)

print("=== PROBABILITY RELIABILITY EVALUATION ===")
print(f"Brier Score Loss: {brier_score:.4f}")

if brier_score < 0.12:
    print("Assessment: Good calibration. The model's probabilities are realistic.")
else:
    print("Assessment: Poor calibration. Model probabilities are distorted and should be calibrated.")

=== PROBABILITY RELIABILITY EVALUATION ===
Brier Score Loss: 0.0259
Assessment: Good calibration. The model's probabilities are realistic.


---
## Summary of Key Rules

1. **Avoid blind `.predict()`:** The default 0.50 threshold is arbitrary and rarely optimal for imbalanced or cost-sensitive problems.
2. **Use `.predict_proba()`:** Always extract continuous probabilities, then apply the cutoff that maximizes business value.
3. **ROC-AUC vs. PR-AUC:**
   * Use **ROC-AUC** (`scoring='roc_auc'`) for balanced or moderate class distributions.
   * Use **PR-AUC** (`scoring='average_precision'`) for severely imbalanced data where false alarms matter.
4. **Validation Integrity:** The optimal threshold must be identified on **Validation folds**, never on the test set.

## Here is a code snippet where you can give some range and let the process end up finding the best threshold And applying it to the model.  
Here there may be a doubt about "where is the model?" see the line 6, that model comes from the code cell 1, where we initialized it and the process of applying the threshold to the model is in this below line:
```python 
y_pred_tuned = (y_probs >= best_threshold).astype(int)
```

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, classification_report, confusion_matrix

# 1. Generate probabilities
y_probs = model.predict_proba(X_test)[:, 1]

# 2. Test 100 possible cutoffs purely with math
thresholds = np.linspace(0.1, 0.9, 100)
scores = []

for t in thresholds:
    preds = (y_probs >= t).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, preds, average='binary', zero_division=0
    )
    scores.append(f1)

# 3. Locate the optimal cutoff that maximizes F1-score
best_idx = np.argmax(scores)
best_threshold = thresholds[best_idx]
best_f1 = scores[best_idx]

# 4. Apply the optimal threshold to produce final binary predictions
y_pred_tuned = (y_probs >= best_threshold).astype(int)

# 5. Output proof of application and comparison against default 0.50
cm_default = confusion_matrix(y_test, (y_probs >= 0.50).astype(int))
cm_tuned = confusion_matrix(y_test, y_pred_tuned)

print(f"Optimal Threshold Found: {best_threshold:.2f}")
print(f"Best Validation F1-Score: {best_f1:.4f}\n")

print(f"--- Application Proof (Confusion Matrix @ Cutoff {best_threshold:.2f}) ---")
df_proof = pd.DataFrame(
    cm_tuned,
    index=['Actual Class 0', 'Actual Class 1'],
    columns=['Predicted Class 0', 'Predicted Class 1']
)
print(df_proof)

print("\n--- Detection Proof ---")
print(f"Class 1 detected at default 0.50 cutoff: {cm_default[1, 1]} / {y_test.sum()}")
print(f"Class 1 detected at tuned {best_threshold:.2f} cutoff:   {cm_tuned[1, 1]} / {y_test.sum()}")

Optimal Threshold Found: 0.13
Best Validation F1-Score: 0.8710

--- Application Proof (Confusion Matrix @ Cutoff 0.13) ---
                Predicted Class 0  Predicted Class 1
Actual Class 0                430                  9
Actual Class 1                  7                 54

--- Detection Proof ---
Class 1 detected at default 0.50 cutoff: 46 / 61
Class 1 detected at tuned 0.13 cutoff:   54 / 61


### Things needed to discuss about to get more better understanding about the notebook.

---
## Step 5: Probability Calibration (Can We Trust the Percentages?)

### 1. What Is Probability Calibration?
When you call `model.predict_proba()`, the model returns confidence values between `0.0` and `1.0` (e.g., `0.80`). 

Calibration asks one straightforward question:
> **If the model assigns an 80% probability to 100 different customers, do roughly 80 of them actually churn in the real world?**

* **Well-Calibrated:** A predicted 0.80 means an 80% real-world event rate. The output represents a true statistical probability.
* **Poorly Calibrated (Uncalibrated):** A predicted 0.80 might mean only 50% or 60% of those cases actually happen. The model is **overconfident** or distorted.

---

### 2. Why Are Machine Learning Models Often Uncalibrated?
Different algorithms calculate their raw scores using different mathematics:

* **Logistic Regression:** Usually naturally well-calibrated because its internal loss function (Log-Loss / Cross-Entropy) directly optimizes the likelihood of true probabilities.
* **Decision Trees & Random Forests:** Tend to cluster probabilities away from 0 and 1 because leaf nodes average a small number of samples, creating discrete, stepwise confidence jumps.
* **Gradient Boosting (XGBoost, LightGBM, HistGradientBoosting):** Often pushes predictions toward extreme values (close to 0.0 or 1.0). The model acts 95% confident on samples where the real-world likelihood is only 65%.
* **Naive Bayes:** Almost always severely uncalibrated because its assumption of feature independence pushes probabilities to extreme 0.0001 or 0.9999 boundaries.

---

### 3. Why Calibration Matters for Threshold Tuning
Threshold shifting and cost-benefit calculations depend entirely on **honest confidence scores**:

1. **Financial Decisions Fail:** If you calculate that a customer with a 30% churn risk is worth a $25 retention discount, but your model's "30%" is actually only a 10% real-world risk, you will waste budget sending discounts to customers who were never leaving.
2. **Medical & Risk Decisions Fail:** In credit scoring or healthcare triage, treating an uncalibrated score as a genuine percentage leads to poor risk allocations.

---

### 4. How to Measure Calibration Numerically: The Brier Score
Instead of plotting calibration curves, measure it directly using the **Brier Score Loss** (`brier_score_loss`):

$$\text{Brier Score} = \frac{1}{N} \sum_{i=1}^{N} (P_i - y_i)^2$$

It computes the mean squared difference between predicted probabilities ($P_i$) and the actual binary outcomes ($y_i \in \{0, 1\}$).

* **0.00:** Perfect calibration (every prediction matched the exact probability of reality).
* **< 0.10:** High quality. Probabilities are reliable and can be treated as genuine percentages.
* **0.10 – 0.20:** Acceptable for general ranking and threshold tuning, but slightly distorted.
* **> 0.25:** Poor calibration. A baseline coin-flip model predicting a flat 0.50 on a balanced dataset scores 0.25, so scores above this range mean the probabilities cannot be trusted as real percentages.
